## Dependencies

In [1]:
## libraries
import os
import sys
import pandas as pd
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.evaluators.config import FEAT_X, FEAT_Z, TARGET
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.resampling import logo_cross_valid, kfold_cross_valid

## Initalization

In [2]:
## setup
data = load_processed_data()
models = load_estimators()

## Training

In [3]:
## leave-one-group-out cross validation (domain)
results_dict_domain = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = logo_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        group = "domain",
        n_jobs = -1,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_domain[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


In [4]:
## leave-one-group-out cross validation (discipline)
results_dict_disc = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = logo_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        group = "discipline",
        n_jobs = -1,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_disc[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


In [5]:
## 2-fold cross validation
results_dict_2fold = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = kfold_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        n_splits = 2,
        n_repeats = 30,
        n_jobs = -1,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_2fold[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


In [6]:
## 5-fold cross validation
results_dict_5fold = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = kfold_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        n_splits = 5,
        n_repeats = 30,
        n_jobs = -1,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_5fold[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


In [7]:
## 10-fold cross validation
results_dict_10fold = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = kfold_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        n_splits = 10,
        n_repeats = 30,
        n_jobs = -1,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_10fold[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


## Post-Processing

In [8]:
## convert all results to dataframes
def _to_results_frame(results_dict: dict) -> pd.DataFrame:
    frame = pd.concat(results_dict.values(), ignore_index = True)
    feat = ["model"] + [c for c in frame.columns if c != "model"]
    return frame[feat]

results_data_domain = _to_results_frame(results_dict_domain)
results_data_disc = _to_results_frame(results_dict_disc)
results_data_3fold = _to_results_frame(results_dict_2fold)
results_data_5fold = _to_results_frame(results_dict_5fold)
results_data_10fold = _to_results_frame(results_dict_10fold)

## Results

In [9]:
## mean across all cv methods
pd.concat([
    results_data_domain,
    results_data_disc,
    results_data_3fold,
    results_data_5fold,
    results_data_10fold
], ignore_index = True).mean(numeric_only = True)

vr    0.112680
mv    2.110531
ms    7.544968
ea    1.524305
ei    0.703531
dtype: float64

In [10]:
results_data_domain.mean(numeric_only = True)

vr    0.093333
mv    3.900291
ms    5.411517
ea    0.922109
ei    0.717955
dtype: float64

In [11]:
results_data_disc.mean(numeric_only = True)

vr    0.120000
mv    1.150390
ms    8.007772
ea    1.691969
ei    0.697637
dtype: float64

In [12]:
results_data_3fold.mean(numeric_only = True)

vr    0.082039
mv    0.915811
ms    4.752635
ea    0.706022
ei    0.709423
dtype: float64

In [13]:
results_data_5fold.mean(numeric_only = True)

vr    0.074370
mv    5.100353
ms    7.674414
ea    1.167730
ei    0.740081
dtype: float64

In [14]:
results_data_10fold.mean(numeric_only = True)

vr     0.095370
mv    15.370161
ms     9.305028
ea     1.518555
ei     0.736307
dtype: float64